In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import tiktoken

# 1. CONFIG

GPT2_SMALL_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "embed_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": True,
}


# 2. TOKENIZER (tiktoken)

def get_tokenizer():
    """Returns GPT-2 tokenizer via tiktoken."""
    return tiktoken.get_encoding("gpt2")


# 3. DATASET & DATALOADER

class TextDataset(Dataset):
    """Sliding window dataset that tokenizes raw text with tiktoken."""

    def __init__(self, text: str, tokenizer, max_length: int, stride: int):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk  = token_ids[i          : i + max_length]
            target_chunk = token_ids[i + 1      : i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader(text: str, batch_size: int = 4, max_length: int = 256,
                      stride: int = 128, shuffle: bool = True,
                      drop_last: bool = True, num_workers: int = 0):
    tokenizer = get_tokenizer()
    dataset   = TextDataset(text, tokenizer, max_length, stride)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      drop_last=drop_last, num_workers=num_workers)


# 4. MULTI-HEAD CAUSAL SELF-ATTENTION

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim: int, n_heads: int, context_length: int,
                 dropout: float, qkv_bias: bool = False):
        super().__init__()
        assert embed_dim % n_heads == 0, "embed_dim must be divisible by n_heads"

        self.n_heads   = n_heads
        self.head_dim  = embed_dim // n_heads
        self.embed_dim = embed_dim

        self.W_qkv = nn.Linear(embed_dim, 3 * embed_dim, bias=qkv_bias)
        self.W_out = nn.Linear(embed_dim, embed_dim)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1).bool()
        )

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.W_qkv(x)                          # (B, T, 3C)
        q, k, v = qkv.split(self.embed_dim, dim=-1)  # each (B, T, C)

        def split_heads(t):
            return t.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        q, k, v = split_heads(q), split_heads(k), split_heads(v)

        # Scaled dot-product attention
        scale  = self.head_dim ** -0.5
        scores = (q @ k.transpose(-2, -1)) * scale    # (B, n_heads, T, T)
        scores = scores.masked_fill(self.mask[:T, :T], float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        weights = self.attn_drop(weights)

        out = weights @ v                              # (B, n_heads, T, head_dim)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.W_out(out)
        return self.resid_drop(out)


# 5. FEED-FORWARD NETWORK

class FeedForward(nn.Module):
    def __init__(self, embed_dim: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.GELU(),
            nn.Linear(4 * embed_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


# 6. TRANSFORMER BLOCK


class TransformerBlock(nn.Module):
    def __init__(self, cfg: dict):
        super().__init__()
        self.attn = MultiHeadAttention(
            embed_dim=cfg["embed_dim"],
            n_heads=cfg["n_heads"],
            context_length=cfg["context_length"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.ff   = FeedForward(cfg["embed_dim"], cfg["drop_rate"])
        self.ln1  = nn.LayerNorm(cfg["embed_dim"])
        self.ln2  = nn.LayerNorm(cfg["embed_dim"])

    def forward(self, x):
        x = x + self.attn(self.ln1(x))  # pre-LayerNorm
        x = x + self.ff(self.ln2(x))
        return x


# 7. GPT-2 MODEL

class GPTModel(nn.Module):
    def __init__(self, cfg: dict):
        super().__init__()
        self.tok_emb  = nn.Embedding(cfg["vocab_size"], cfg["embed_dim"])
        self.pos_emb  = nn.Embedding(cfg["context_length"], cfg["embed_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.blocks   = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        self.ln_f     = nn.LayerNorm(cfg["embed_dim"])
        self.head     = nn.Linear(cfg["embed_dim"], cfg["vocab_size"], bias=False)


        self.head.weight = self.tok_emb.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx):
        B, T = idx.shape
        device = idx.device

        tok = self.tok_emb(idx)
        pos = self.pos_emb(torch.arange(T, device=device))
        x   = self.drop_emb(tok + pos)
        x   = self.blocks(x)
        x   = self.ln_f(x)
        return self.head(x)                            # (B, T, vocab_size)


# 8. TEXT GENERATION UTILITY

@torch.no_grad()
def generate(model, idx, max_new_tokens: int, context_size: int,
             temperature: float = 1.0, top_k: int = None):
    """Autoregressive token generation with temperature and top-k sampling."""
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        logits   = model(idx_cond)[:, -1, :]          # (B, vocab_size)

        if temperature != 1.0:
            logits = logits / temperature
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float("-inf")

        probs    = torch.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx      = torch.cat([idx, idx_next], dim=1)
    return idx


def text_to_ids(text: str, tokenizer, device):
    return torch.tensor(tokenizer.encode(text)).unsqueeze(0).to(device)


def ids_to_text(ids: torch.Tensor, tokenizer):
    return tokenizer.decode(ids.squeeze(0).tolist())



# QUICK SMOKE-TEST


if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")

    cfg   = GPT2_SMALL_CONFIG
    model = GPTModel(cfg).to(device)
    total = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {total:,}")

    tokenizer = get_tokenizer()
    sample    = "The transformer architecture revolutionized NLP"
    ids       = text_to_ids(sample, tokenizer, device)
    out       = model(ids)
    print(f"Forward pass output shape: {out.shape}")   # (1, T, 50257)

    generated = generate(model, ids, max_new_tokens=20,
                         context_size=cfg["context_length"],
                         temperature=0.8, top_k=40)
    print("Generated:", ids_to_text(generated, tokenizer))
    print("Stage 1 complete.")

Device: cpu
Model parameters: 124,439,808
Forward pass output shape: torch.Size([1, 7, 50257])
Generated: The transformer architecture revolutionized NLP6000 Next arri Hunterabsolutely217 GloriaDiscuss Awards Next fingers Wed impair815esm Indokaya217 planting planting
Stage 1 complete.


In the colab terminal run these Commands



```
sudo apt install zstd
curl -fsSL https://ollama.com/install.sh -o install.sh
cat install.sh
sh install.sh
```



In [ ]:
"""
Stage 2: Training Loop, Model Evaluation, Load GPT-2 Small Pretrained Weights
- AdamW optimiser with cosine LR schedule
- Cross-entropy loss + perplexity
- Load GPT-2 Small weights from HuggingFace
- Save / load checkpoints
- Evaluate with Ollama
"""

import os
import math
import json
import time
import torch
import torch.nn as nn
import numpy as np
import tiktoken
import requests
from torch.utils.data import DataLoader
# from stage1_architecture import (
#     GPTModel, GPT2_SMALL_CONFIG, get_tokenizer,
#     create_dataloader, generate, text_to_ids, ids_to_text
# )

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# ──────────────────────────────────────────────
# 1. LOSS HELPERS
# ──────────────────────────────────────────────

def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch  = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)                         # (B, T, V)
    loss   = nn.functional.cross_entropy(
        logits.flatten(0, 1), target_batch.flatten()   # (B*T, V), (B*T,)
    )
    return loss


@torch.no_grad()
def calc_loss_loader(data_loader, model, device, num_batches=None):
    model.eval()
    total_loss, count = 0.0, 0
    for i, (x, y) in enumerate(data_loader):
        if num_batches is not None and i >= num_batches:
            break
        total_loss += calc_loss_batch(x, y, model, device).item()
        count += 1
    return total_loss / max(count, 1)


def perplexity(loss: float) -> float:
    return math.exp(loss)


# ──────────────────────────────────────────────
# 2. COSINE LR SCHEDULE
# ──────────────────────────────────────────────

def get_lr(step: int, warmup_steps: int, max_steps: int,
           max_lr: float, min_lr: float) -> float:
    if step < warmup_steps:
        return max_lr * step / warmup_steps
    if step >= max_steps:
        return min_lr
    progress = (step - warmup_steps) / (max_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))


# ──────────────────────────────────────────────
# 3. TRAINING LOOP
# ──────────────────────────────────────────────

def train_model(
    model,
    train_loader: DataLoader,
    val_loader: DataLoader,
    n_epochs: int = 10,
    eval_freq: int = 200,
    eval_iters: int = 20,
    save_path: str = "gpt2_checkpoint.pth",
    max_lr: float = 6e-4,
    min_lr: float = 6e-5,
    warmup_steps: int = 200,
    grad_clip: float = 1.0,
):
    model.to(DEVICE)
    optimiser   = torch.optim.AdamW(model.parameters(), lr=max_lr,
                                     betas=(0.9, 0.95), weight_decay=0.1)
    tokenizer   = get_tokenizer()
    global_step = 0
    history     = {"train_loss": [], "val_loss": [], "steps": []}

    total_steps = n_epochs * len(train_loader)
    print(f"Training for {n_epochs} epoch(s) | {total_steps} total steps")
    print(f"Device: {DEVICE}\n")

    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0.0

        for batch_idx, (x, y) in enumerate(train_loader):
            # LR schedule
            lr = get_lr(global_step, warmup_steps, total_steps, max_lr, min_lr)
            for pg in optimiser.param_groups:
                pg["lr"] = lr

            optimiser.zero_grad()
            loss = calc_loss_batch(x, y, model, DEVICE)
            loss.backward()

            # Gradient clipping
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimiser.step()

            epoch_loss  += loss.item()
            global_step += 1

            # Periodic evaluation
            if global_step % eval_freq == 0 or global_step == 1:
                train_loss = calc_loss_loader(train_loader, model, DEVICE, eval_iters)
                val_loss   = calc_loss_loader(val_loader,   model, DEVICE, eval_iters)
                history["train_loss"].append(train_loss)
                history["val_loss"].append(val_loss)
                history["steps"].append(global_step)
                model.train()
                print(
                    f"Ep {epoch+1}/{n_epochs} | Step {global_step:>6} | "
                    f"LR {lr:.2e} | Train loss {train_loss:.4f} "
                    f"(PPL {perplexity(train_loss):.1f}) | "
                    f"Val loss {val_loss:.4f} (PPL {perplexity(val_loss):.1f})"
                )

        avg_epoch = epoch_loss / len(train_loader)
        print(f"\n── End of epoch {epoch+1}: avg train loss = {avg_epoch:.4f} ──\n")

    # Save checkpoint
    save_checkpoint(model, optimiser, global_step, history, save_path)
    return history


# ──────────────────────────────────────────────
# 4. CHECKPOINT SAVE / LOAD
# ──────────────────────────────────────────────

def save_checkpoint(model, optimiser, step, history, path: str):
    checkpoint = {
        "model_state":     model.state_dict(),
        "optimiser_state": optimiser.state_dict(),
        "step":            step,
        "history":         history,
        "config":          GPT2_SMALL_CONFIG,
    }
    torch.save(checkpoint, path)
    print(f"Checkpoint saved → {path}")


def load_checkpoint(path: str, device=DEVICE):
    checkpoint = torch.load(path, map_location=device)
    model = GPTModel(checkpoint["config"]).to(device)
    model.load_state_dict(checkpoint["model_state"])
    print(f"Checkpoint loaded ← {path}  (step {checkpoint['step']})")
    return model, checkpoint


# ──────────────────────────────────────────────
# 5. LOAD GPT-2 SMALL PRETRAINED WEIGHTS
# ──────────────────────────────────────────────

def load_gpt2_pretrained(model: GPTModel) -> GPTModel:
    """
    Transfer GPT-2 Small weights from HuggingFace transformers into our model.
    Requires: pip install transformers
    """
    try:
        from transformers import GPT2Model as HF_GPT2
    except ImportError:
        raise ImportError("Run: pip install transformers")

    print("Downloading GPT-2 Small pretrained weights…")
    hf_model = HF_GPT2.from_pretrained("gpt2")
    hf_sd    = hf_model.state_dict()

    our_sd   = model.state_dict()

    # ── Embedding layers ──
    our_sd["tok_emb.weight"].copy_(hf_sd["wte.weight"])
    our_sd["pos_emb.weight"].copy_(hf_sd["wpe.weight"])

    # ── Transformer blocks ──
    for layer_idx in range(GPT2_SMALL_CONFIG["n_layers"]):
        hf_pfx = f"h.{layer_idx}"
        ou_pfx = f"blocks.{layer_idx}"

        # Layer norms
        our_sd[f"{ou_pfx}.ln1.weight"].copy_(hf_sd[f"{hf_pfx}.ln_1.weight"])
        our_sd[f"{ou_pfx}.ln1.bias"  ].copy_(hf_sd[f"{hf_pfx}.ln_1.bias"])
        our_sd[f"{ou_pfx}.ln2.weight"].copy_(hf_sd[f"{hf_pfx}.ln_2.weight"])
        our_sd[f"{ou_pfx}.ln2.bias"  ].copy_(hf_sd[f"{hf_pfx}.ln_2.bias"])

        # QKV weight and bias (HF stores them as separate q/k/v, GPT-2 as c_attn)
        c_attn_w = hf_sd[f"{hf_pfx}.attn.c_attn.weight"].T  # (C, 3C) → (3C, C)
        c_attn_b = hf_sd[f"{hf_pfx}.attn.c_attn.bias"]
        our_sd[f"{ou_pfx}.attn.W_qkv.weight"].copy_(c_attn_w)
        our_sd[f"{ou_pfx}.attn.W_qkv.bias"  ].copy_(c_attn_b)

        # Output projection
        our_sd[f"{ou_pfx}.attn.W_out.weight"].copy_(
            hf_sd[f"{hf_pfx}.attn.c_proj.weight"].T)
        our_sd[f"{ou_pfx}.attn.W_out.bias"].copy_(
            hf_sd[f"{hf_pfx}.attn.c_proj.bias"])

        # MLP
        our_sd[f"{ou_pfx}.ff.net.0.weight"].copy_(
            hf_sd[f"{hf_pfx}.mlp.c_fc.weight"].T)
        our_sd[f"{ou_pfx}.ff.net.0.bias"  ].copy_(
            hf_sd[f"{hf_pfx}.mlp.c_fc.bias"])
        our_sd[f"{ou_pfx}.ff.net.2.weight"].copy_(
            hf_sd[f"{hf_pfx}.mlp.c_proj.weight"].T)
        our_sd[f"{ou_pfx}.ff.net.2.bias"  ].copy_(
            hf_sd[f"{hf_pfx}.mlp.c_proj.bias"])

    # ── Final LayerNorm ──
    our_sd["ln_f.weight"].copy_(hf_sd["ln_f.weight"])
    our_sd["ln_f.bias"  ].copy_(hf_sd["ln_f.bias"])

    model.load_state_dict(our_sd)
    print("GPT-2 Small weights loaded successfully.")
    return model


# ──────────────────────────────────────────────
# 6. EVALUATE USING OLLAMA
# ──────────────────────────────────────────────

def evaluate_with_ollama(
    model,
    prompts: list[str],
    reference_answers: list[str] | None = None,
    ollama_model: str = "llama3",
    ollama_url: str = "http://localhost:11434/api/generate",
    temperature: float = 0.7,
    top_k: int = 40,
    max_new_tokens: int = 200,
):
    """
    Generate answers from our model and optionally send them to Ollama
    for quality scoring against reference answers.

    Requirements: Ollama running locally  →  ollama serve
    """
    tokenizer = get_tokenizer()
    results   = []

    for idx, prompt in enumerate(prompts):
        print(f"\n── Prompt {idx+1}: {prompt[:80]}…")

        # Our model generates a response
        ids      = text_to_ids(prompt, tokenizer, DEVICE)
        gen_ids  = generate(model, ids, max_new_tokens=max_new_tokens,
                            context_size=GPT2_SMALL_CONFIG["context_length"],
                            temperature=temperature, top_k=top_k)
        our_resp = ids_to_text(gen_ids, tokenizer)
        print(f"Our model: {our_resp[:300]}")

        # Optionally ask Ollama to judge quality
        ollama_score = None
        if reference_answers and idx < len(reference_answers):
            judge_prompt = (
                f"Reference answer:\n{reference_answers[idx]}\n\n"
                f"Model answer:\n{our_resp}\n\n"
                "Rate the model answer for accuracy and relevance (1-10). "
                "Reply with just a JSON object: {\"score\": <int>, \"reason\": \"<short reason>\"}"
            )
            try:
                payload  = {"model": ollama_model, "prompt": judge_prompt,
                            "stream": False}
                response = requests.post(ollama_url, json=payload, timeout=60)
                raw      = response.json().get("response", "")
                parsed   = json.loads(raw)
                ollama_score = parsed
                print(f"Ollama score: {ollama_score}")
            except Exception as e:
                print(f"Ollama eval skipped ({e})")

        results.append({
            "prompt":       prompt,
            "our_response": our_resp,
            "ollama_score": ollama_score,
        })

    return results



# DEMO ENTRY POINT
# ──────────────────────────────────────────────

if __name__ == "__main__":
    CHECKPOINT = "gpt2_checkpoint.pth"

    # ── Build or load model ──
    if os.path.exists(CHECKPOINT):
        model, ckpt = load_checkpoint(CHECKPOINT)
    else:
        model = GPTModel(GPT2_SMALL_CONFIG).to(DEVICE)
        model = load_gpt2_pretrained(model)

        # ── Dummy training data (replace with real corpus) ──
        SAMPLE_TEXT = (
            "Transformers are a type of neural network architecture that uses "
            "self-attention mechanisms. They were introduced in the paper "
            "'Attention Is All You Need' by Vaswani et al. in 2017. "
            "The architecture consists of encoder and decoder stacks, each "
            "containing multi-head attention and feed-forward layers. "
            "GPT models use only the decoder portion of the transformer. "
        ) * 400  # replicate for demo purposes

        train_loader = create_dataloader(SAMPLE_TEXT, batch_size=4,
                                         max_length=256, stride=128, shuffle=True)
        val_loader   = create_dataloader(SAMPLE_TEXT, batch_size=4,
                                         max_length=256, stride=256, shuffle=False)

        history = train_model(
            model, train_loader, val_loader,
            n_epochs=2, eval_freq=50,
            save_path=CHECKPOINT,
        )

    # ── Quick generation test ──
    tokenizer = get_tokenizer()
    prompt    = "Attention mechanisms in neural networks"
    ids       = text_to_ids(prompt, tokenizer, DEVICE)
    out_ids   = generate(model, ids, max_new_tokens=60,
                         context_size=GPT2_SMALL_CONFIG["context_length"],
                         temperature=0.8, top_k=40)
    print("\nGenerated text:")
    print(ids_to_text(out_ids, tokenizer))

    # ── Ollama evaluation
    eval_prompts = [
        "What is the attention mechanism in transformers?",
        "Explain GPT-2 architecture briefly.",
    ]
    evaluate_with_ollama(model, eval_prompts, max_new_tokens=100)
    print("\nStage 2 complete.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT-2 Small weights loaded successfully.
Training for 2 epoch(s) | 118 total steps
Device: cpu

Ep 1/2 | Step      1 | LR 0.00e+00 | Train loss 1.2660 (PPL 3.5) | Val loss 1.2689 (PPL 3.6)
Ep 1/2 | Step     50 | LR 1.47e-04 | Train loss 0.0112 (PPL 1.0) | Val loss 0.0091 (PPL 1.0)

── End of epoch 1: avg train loss = 0.3567 ──

Ep 2/2 | Step    100 | LR 2.97e-04 | Train loss 0.0045 (PPL 1.0) | Val loss 0.0064 (PPL 1.0)

── End of epoch 2: avg train loss = 0.0132 ──

Checkpoint saved → gpt2_checkpoint.pth

Generated text:
Attention mechanisms in neural networks architecture that uses self-attention mechanisms. They were introduced in the paper 'Attention Is All You Need' by Vaswani et al. in 2017. The architecture consists of encoder and decoder stacks, each containing multi-head attention and feed-forward layers. GPT models use

── Prompt 1: What is the attention mechanism in transformers?…
Our model: What is the attention mechanism in transformers? They were introduced in the paper 'A